# Análise Exploratória dos Dados

## Alfa de Cronbach

In [43]:
from pathlib import Path

import numpy as np
import pandas as pd

base = Path.cwd().parent / 'dados'
df = pd.read_csv(base / 'amostra_v2.csv')

colunas_numericas = [col for col in df.columns if col not in {'ID', 'Município', 'UF'}]
for col in colunas_numericas:
    df[col] = df[col].astype(str).str.replace(',', '.', regex=False)
    df[col] = pd.to_numeric(df[col], errors='coerce')

def alfa_cronbach_padronizado(dados: pd.DataFrame) -> float:
    dados = dados.dropna(axis=0, how='any')
    k = dados.shape[1]
    if k < 2:
        return np.nan

    matriz_corr = dados.corr().to_numpy()
    mascara = ~np.eye(k, dtype=bool)
    correlacoes = matriz_corr[mascara]
    correlacoes = correlacoes[~np.isnan(correlacoes)]

    if correlacoes.size == 0:
        return np.nan

    r_barra = correlacoes.mean()
    return float((k * r_barra) / (1 + (k - 1) * r_barra))

resultados = []
alpha_total = alfa_cronbach_padronizado(df[colunas_numericas])
resultados.append({
    'cenario': 'dataset completo',
    'variavel_removida': None,
    'n_variaveis': len(colunas_numericas),
    'alfa_padronizado': alpha_total,
})

for coluna_removida in colunas_numericas:
    subconjunto = [col for col in colunas_numericas if col != coluna_removida]
    alpha = alfa_cronbach_padronizado(df[subconjunto])
    resultados.append({
        'cenario': 'sem uma variável',
        'variavel_removida': coluna_removida,
        'n_variaveis': len(subconjunto),
        'alfa_padronizado': alpha,
    })

tabela_resultados = pd.DataFrame(resultados)
tabela_resultados = tabela_resultados.sort_values(
    by=['cenario', 'alfa_padronizado'],
    ascending=[True, False],
)

display(tabela_resultados)
print(f'Alpha padronizado do dataset completo: {alpha_total:.4f}')

,cenario,variavel_removida,n_variaveis,alfa_padronizado
0,dataset completo,NaN,4,0.455972
2,sem uma variável,areas_verdes_urbanas,3,0.732947
3,sem uma variável,focos_calor,3,0.282436
4,sem uma variável,emissao_co2_percapita,3,0.158065
1,sem uma variável,distancia_capital_regional_km,3,0.111878


Alpha padronizado do dataset completo: 0.4560


## Teste de Aleatoriedade

In [44]:
from pathlib import Path

import numpy as np
import pandas as pd
from math import erf, sqrt

base = Path.cwd().parent / 'dados'
df = pd.read_csv(base / 'amostra_v2.csv')

colunas_numericas = [col for col in df.columns if col not in {'ID', 'Município', 'UF'}]
for col in colunas_numericas:
    df[col] = df[col].astype(str).str.replace(',', '.', regex=False)
    df[col] = pd.to_numeric(df[col], errors='coerce')


def teste_runs_serie(serie: pd.Series) -> dict:
    serie = serie.dropna()
    if serie.nunique() < 2:
        return {
            'n': int(serie.shape[0]),
            'runs': np.nan,
            'z': np.nan,
            'p_valor': np.nan,
            'conclusao': 'sem variabilidade suficiente',
        }

    mediana = serie.median()
    sequencia = serie.map(lambda valor: 1 if valor >= mediana else 0)
    sequencia = sequencia[sequencia.notna()].to_numpy()

    if len(sequencia) < 2:
        return {
            'n': int(len(sequencia)),
            'runs': np.nan,
            'z': np.nan,
            'p_valor': np.nan,
            'conclusao': 'amostra insuficiente',
        }

    positivos = int(sequencia.sum())
    negativos = int(len(sequencia) - positivos)

    if positivos == 0 or negativos == 0:
        return {
            'n': int(len(sequencia)),
            'runs': np.nan,
            'z': np.nan,
            'p_valor': np.nan,
            'conclusao': 'todos os valores caem no mesmo lado da mediana',
        }

    runs = 1 + int(np.sum(sequencia[1:] != sequencia[:-1]))
    n1 = positivos
    n2 = negativos
    n = n1 + n2
    media_runs = (2 * n1 * n2) / n + 1
    variancia_runs = (2 * n1 * n2 * (2 * n1 * n2 - n)) / (n**2 * (n - 1))

    if variancia_runs <= 0:
        return {
            'n': int(n),
            'runs': runs,
            'z': np.nan,
            'p_valor': np.nan,
            'conclusao': 'variância nula no teste',
        }

    z = (runs - media_runs) / sqrt(variancia_runs)
    p_valor = 2 * (1 - 0.5 * (1 + erf(abs(z) / sqrt(2))))

    return {
        'n': int(n),
        'runs': int(runs),
        'z': float(z),
        'p_valor': float(p_valor),
        'conclusao': 'sequência aleatória' if p_valor >= 0.05 else 'sequência não aleatória',
    }

resultados_runs = []
for coluna in colunas_numericas:
    resultado = teste_runs_serie(df[coluna])
    resultado['variavel'] = coluna
    resultados_runs.append(resultado)

tabela_runs = pd.DataFrame(resultados_runs)
tabela_runs = tabela_runs[['variavel', 'n', 'runs', 'z', 'p_valor', 'conclusao']].sort_values('p_valor', ascending=True)

display(tabela_runs)
print('Teste de aleatoriedade concluído com p-valor bilateral por variável numérica.')

,variavel,n,runs,z,p_valor,conclusao
3,emissao_co2_percapita,100,46,-1.005089,0.314854,sequência aleatória
0,distancia_capital_regional_km,100,55,0.804071,0.421356,sequência aleatória
1,areas_verdes_urbanas,100,54,0.603053,0.546473,sequência aleatória
2,focos_calor,100,53,0.402036,0.687658,sequência aleatória


Teste de aleatoriedade concluído com p-valor bilateral por variável numérica.


## Cálculo do Tamanho da Amostra

In [45]:
from pathlib import Path

import numpy as np
import pandas as pd

base = Path.cwd().parent / 'dados'
df = pd.read_csv(base / 'amostra_v2.csv')

colunas_numericas = [col for col in df.columns if col not in {'ID', 'Município', 'UF'}]
for col in colunas_numericas:
    df[col] = df[col].astype(str).str.replace(',', '.', regex=False)
    df[col] = pd.to_numeric(df[col], errors='coerce')

N = 786
z = 1.96

erros_desejados = {
    'distancia_capital_regional_km': 17,
    'areas_verdes_urbanas': 0.55,
    'emissao_co2_percapita': 1.6,
    'focos_calor': 1.8,
}

res = []
for col in colunas_numericas:
    x = df[col].dropna()
    var = float(x.var(ddof=1))
    E_desejado = erros_desejados.get(col, np.nan)

    if np.isnan(E_desejado) or var <= 0:
        n_calc = np.nan
        n_obs = np.nan
        complemento = np.nan
        erro_obt = np.nan
    else:
        n0 = (z**2 * var) / (E_desejado**2)
        n_calc = (N * n0) / (N + n0)
        n_obs = int(np.ceil(n_calc))
        complemento = N - n_obs
        erro_obt = float(np.sqrt(((N - n_obs) / (N - 1)) * (var / n_obs)))

    res.append({
        'variavel': col,
        'variancia': var,
        'n': n_obs,
        'complemento_n': complemento,
        'erro_desejado': E_desejado,
        'erro_obtido': erro_obt,
    })

tabela_n = pd.DataFrame(res)
tabela_n = tabela_n[['variavel', 'variancia', 'n', 'complemento_n', 'erro_desejado', 'erro_obtido']]
tabela_n['variancia'] = tabela_n['variancia'].round(4)
tabela_n['erro_desejado'] = tabela_n['erro_desejado'].round(4)
tabela_n['erro_obtido'] = tabela_n['erro_obtido'].round(4)

display(tabela_n)
print('População total:', N)
print('Confiança:', '95%')

,variavel,variancia,n,complemento_n,erro_desejado,erro_obtido
0,distancia_capital_regional_km,8219.4622,96,690,17.00,8.6751
1,areas_verdes_urbanas,8.9383,100,686,0.55,0.2795
2,focos_calor,90.2515,95,691,1.80,0.9145
3,emissao_co2_percapita,74.3846,98,688,1.60,0.8156


População total: 786
Confiança: 95%


## Tratamento dos Dados

### Tratando inconsistência de separadores decimais